In [1]:
import fitz  # PyMuPDF

# Open a PDF
doc = fitz.open("/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/sample_for_reducto.pdf")

# Loop through all pages
for page_num in range(len(doc)):
    page = doc[page_num]
    print(f"\n--- Page {page_num+1} ---")

    # 1) Simple plain text
    text = page.get_text("text")
    print("Plain Text:\n", text)

    # 2) Text blocks with coordinates
    blocks = page.get_text("blocks")  # list of (x0, y0, x1, y1, text, block_no, block_type)
    for b in blocks:
        print(f"Block: {b[4]} (coords: {b[:4]})")

    # 3) Extract words
    words = page.get_text("words")  # list of (x0, y0, x1, y1, word, block_no, line_no, word_no)
    print("Words on page:", [w[4] for w in words])

    # 4) Extract in JSON-like dict format
    json_data = page.get_text("dict")
    print("JSON keys:", json_data.keys())
    # json_data["blocks"] gives layout info



--- Page 1 ---
Plain Text:
 If “Yes” enter total number of accounts  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
No
Yes
14a Does the filer have a financial interest in 25 or more financial accounts?
13 Country  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
11 State . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
City  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
10
Zip/postal code . . . . . . . . . . . . . . . . 

In [5]:
import fitz, json, pathlib

def capture_page(page):
    # 1) Rich text dict (blocks -> lines -> spans)
    d = page.get_text("dict")

    # 2) Built-in tables (if any)
    tables_out = []
    try:
        tb = page.find_tables()
        for t in getattr(tb, "tables", []):
            rows = t.extract()  # list[list[str|None]]
            cells = []
            for r, row in enumerate(rows):
                for c, cell in enumerate(row):
                    if cell is None: 
                        continue
                    cells.append({"row": r, "col": c, "text": cell})
            tables_out.append({"bbox": list(t.bbox), "cells": cells})
    except Exception:
        pass

    # 3) Anchors you care about (literal text anchors)
    # Tip: add more anchors here as needed
    anchors = {}
    for label in [
        "FIRST NAME", "MIDDLE INITIAL", "LAST NAME OR ORGANIZATION NAME",
        "INDIVIDUAL", "U.S. TAXPAYER IDENTIFICATION NUMBER",
        "TIN TYPE", "TYPE OF FILER", "MAILING ADDRESS",
        "CITY", "STATE", "ZIP/POSTAL CODE", "COUNTRY",
        "AMENDED", "PRIOR REPORT BSA IDENTIFIER",
        "THIS REPORT IS FOR CALENDAR YEAR ENDED",
        "Yes", "No", "X", "Passport"
    ]:
        rects = [list(r) for r in page.search_for(label)]
        if rects:
            anchors[label] = rects

    return {"text_dict": d, "tables": tables_out, "anchors": anchors}

def capture_pdf(path):
    doc = fitz.open(path)
    out = {
        "doc_id": pathlib.Path(path).name,
        "meta": {"num_pages": len(doc)},
        "pages": []
    }
    for i, p in enumerate(doc):
        out["pages"].append(capture_page(p))
    return out

if __name__ == "__main__":
    pdf = "/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/sample_for_reducto.pdf"
    data = capture_pdf(pdf)
    print(json.dumps(data, indent=2))
    # dump to a file (use .json extension ideally)
    out_path = "/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/response.json"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


{
  "doc_id": "sample_for_reducto.pdf",
  "meta": {
    "num_pages": 1
  },
  "pages": [
    {
      "text_dict": {
        "width": 612.0,
        "height": 792.0,
        "blocks": [
          {
            "number": 0,
            "type": 0,
            "bbox": [
              40.15275955200195,
              456.07098388671875,
              476.3701477050781,
              476.6495666503906
            ],
            "lines": [
              {
                "spans": [
                  {
                    "size": 7.679999828338623,
                    "flags": 4,
                    "bidi": 0,
                    "char_flags": 16,
                    "font": "ArialMT",
                    "color": 0,
                    "alpha": 255,
                    "ascender": 0.9049999713897705,
                    "descender": -0.21199999749660492,
                    "text": "If \u201cYes\u201d enter total number of accounts",
                    "origin": [
                      140.9

In [7]:
import re, math, json
from typing import Dict, Any, List, Optional, Tuple

# ---------- label map → final keys ----------
LABELS = [
    (r"\bTHIS REPORT IS FOR CALENDAR YEAR ENDED\b", "report_year", None),
    (r"\bTYPE OF FILER\b", "filer_type", None),
    (r"\bU\.S\. TAXPAYER IDENTIFICATION NUMBER\b", "tin_value", "ssn"),
    (r"\bTIN TYPE\b", "tin_type", None),
    (r"\bFOREIGN IDENTIFICATION\b", None, None),  # section header only
    (r"\b4A\b.*\bTYPE\b", "foreign_id_type", None),
    (r"\b4B\b.*\bNUMBER\b", "foreign_id_number", "passport"),
    (r"\b4C\b.*\bCOUNTRY OF ISSUE\b", "foreign_id_country", None),
    (r"\bINDIVIDUAL[’']S DATE OF BIRTH\b", "dob", "date"),
    (r"\bFIRST NAME\b", "first_name", None),
    (r"\bMIDDLE INITIAL\b", "middle_initial", None),
    (r"\bLAST NAME OR ORGANIZATION NAME\b", "last_or_org", None),
    (r"\bMAILING ADDRESS\b", "address_line1", None),
    (r"\bCITY\b", "city", None),
    (r"\bSTATE\b", "state", None),
    (r"ZIP/POSTAL CODE", "postal", "zip"),
    (r"\bCOUNTRY\b", "country", None),
    (r"\bAMENDED\b", "amended", None),
    (r"\b14A\b.*25 OR MORE", "q14a_answer", None),
    (r"\b14B\b.*25 OR MORE", "q14b_answer", None),
]

RX = {
    "date": re.compile(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b"),
    "ssn": re.compile(r"\b\d{3}-?\d{2}-?\d{4}\b"),
    "zip": re.compile(r"\b\d{5}(?:-\d{4})?\b"),
    "passport": re.compile(r"\b[0-9A-Z]{6,15}\b"),
}

DOTS = re.compile(r"\s(\.[\s\.]*){3,}")

def clean_text(s: str) -> str:
    s = DOTS.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def lines_from_page(text_dict: Dict[str, Any]) -> List[Dict[str, Any]]:
    out = []
    for b in text_dict.get("blocks", []):
        for ln in b.get("lines", []):
            txt = clean_text("".join(sp.get("text", "") for sp in ln.get("spans", [])))
            if txt:
                out.append({"text": txt, "bbox": ln.get("bbox", [0,0,0,0])})
    return out

def y_center(bb): return (bb[1]+bb[3])/2.0
def same_band(a,b,y_tol=8): return abs(y_center(a)-y_center(b)) <= y_tol

def nearest_right(anchor_ln, lines, max_dx=220, y_tol=8):
    ax1 = anchor_ln["bbox"][2]
    best, bestd = None, 1e9
    for ln in lines:
        x0 = ln["bbox"][0]
        if x0 >= ax1 and same_band(anchor_ln["bbox"], ln["bbox"], y_tol):
            d = x0 - ax1
            if 0 <= d <= max_dx and d < bestd:
                best, bestd = ln, d
    return best

def nearest_below(anchor_ln, lines, max_dy=40):
    ay1 = anchor_ln["bbox"][3]
    best, bestd = None, 1e9
    for ln in lines:
        dy = ln["bbox"][1] - ay1
        if 0 <= dy <= max_dy:
            d = math.hypot((ln["bbox"][0]-anchor_ln["bbox"][0]), dy)
            if d < bestd:
                best, bestd = ln, d
    return best

def label_hits(lines):
    hits = []
    for pat, key, hint in LABELS:
        rx = re.compile(pat, re.I)
        for ln in lines:
            if rx.search(ln["text"]):
                hits.append((key, hint, ln))
    return hits

def confidence(anchor_ln, value_ln, hint):
    if not value_ln: 
        return 0.2
    base = 0.5
    band = 0.2 if same_band(anchor_ln["bbox"], value_ln["bbox"]) else 0.0
    d = max(1.0, value_ln["bbox"][0] - anchor_ln["bbox"][2])
    dist = min(0.2, 60.0 / (60.0 + d))
    rx_bonus = 0.0
    if hint and hint in RX and RX[hint].search(value_ln["text"]):
        rx_bonus = 0.1
    return round(base + band + dist + rx_bonus, 2)

def merge_address_lines(current_value, new_text):
    # simple heuristic: if value exists, append separated by space (or newline if short)
    if not current_value:
        return new_text
    if len(new_text) <= 10 or re.search(r"\b(apt|unit|ste|#)\b", new_text, re.I):
        return current_value + " " + new_text
    return current_value + " " + new_text

def reduce_page(text_dict, include_provenance=False) -> Dict[str, Any]:
    lines = lines_from_page(text_dict)
    out: Dict[str, Any] = {}

    hits = label_hits(lines)
    for key, hint, a_ln in hits:
        if key is None:
            continue  # section header only
        cand = nearest_right(a_ln, lines) or nearest_below(a_ln, lines)
        conf = confidence(a_ln, cand, hint)
        val  = cand["text"] if cand else None

        # special cases
        if key == "report_year" and val:
            # often like "... 12/31/"; pull year from any nearby bold year on page later if needed
            # fallback: scan all lines for lone 4-digit year
            m = re.search(r"\b(19|20)\d{2}\b", " ".join(ln["text"] for ln in lines))
            if m: val = m.group(0)
        if key == "amended" and val:
            # this is a label; true value is presence of an 'X' near it; default to "No"
            # (you can improve by scanning for small 'X' tokens)
            val = "No"

        if key == "address_line1" and val:
            # try to chain the next 1–2 stacked short lines (apt, etc.)
            below = nearest_below(cand, lines, max_dy=18)
            if below and below["text"] and len(below["text"]) <= 20:
                val = merge_address_lines(val, below["text"])

        rec = {"value": val, "confidence": conf}
        if include_provenance:
            rec.update({"key_bbox": a_ln["bbox"], "value_bbox": (cand["bbox"] if cand else None)})
        # keep best
        if key not in out or conf > out[key]["confidence"]:
            out[key] = rec

    # Checkbox questions (14a/14b) – quick heuristic:
    text_all = " ".join(ln["text"] for ln in lines)
    # If we see a solitary 'X' nearer to Yes/No you can wire a fuller detector; minimal default:
    for qk in ("q14a_answer","q14b_answer"):
        if qk in out and not out[qk]["value"]:
            out[qk] = {"value": "No", "confidence": 0.60}

    return out

def reduce_document(capture: Dict[str, Any], include_provenance=False) -> Dict[str, Any]:
    merged: Dict[str, Any] = {}
    for page in capture["pages"]:
        page_out = reduce_page(page["text_dict"], include_provenance=include_provenance)
        for k, v in page_out.items():
            if k not in merged or v["confidence"] > merged[k]["confidence"]:
                merged[k] = v
    return merged

# --- Example run ---
if __name__ == "__main__":
    # load your capture JSON (output of capture_pdf)
    with open("/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/response.json","r") as f:
        capture = json.load(f)
    result = reduce_document(capture, include_provenance=False)
    print(json.dumps(result, indent=2))


{
  "report_year": {
    "value": "2024",
    "confidence": 0.9
  },
  "filer_type": {
    "value": "Individual",
    "confidence": 0.9
  },
  "tin_value": {
    "value": "123654987",
    "confidence": 1.0
  },
  "tin_type": {
    "value": "SSN/ITIN",
    "confidence": 0.9
  },
  "foreign_id_type": {
    "value": "Passport",
    "confidence": 0.9
  },
  "foreign_id_country": {
    "value": "Falkland Islands (Isla",
    "confidence": 0.9
  },
  "dob": {
    "value": "06/06/1979",
    "confidence": 1.0
  },
  "first_name": {
    "value": "Fname",
    "confidence": 0.9
  },
  "middle_initial": {
    "value": "Mailing address",
    "confidence": 0.7
  },
  "last_or_org": {
    "value": "GenInfo",
    "confidence": 0.9
  },
  "address_line1": {
    "value": "444 mainstreet MI",
    "confidence": 0.9
  },
  "city": {
    "value": "Iron Mountain",
    "confidence": 0.9
  },
  "state": {
    "value": "MI",
    "confidence": 0.9
  },
  "postal": {
    "value": "42929",
    "confidence": 1.0
  }

In [8]:
"""
Form reducer for PyMuPDF capture → human-readable JSON.

Pipeline:
  1) Start from capture_json = capture_pdf(pdf)  # your existing function
  2) reduce_document(capture_json, include_provenance=False, include_descriptions=True)
     -> {"first_name": {"value": "John", "confidence": 0.92, "description": "..."},
         ...}
  3) (Optional) write_pdf_description("summary.pdf", reduced_json)

Design:
  - LABELS: regex → canonical key → value-hint (regex validator)
  - We search for label lines, then pick the nearest right/below value line.
  - Confidence is transparent: base + band bonus + distance bonus + regex bonus.
"""

import re, math, json
from typing import Dict, Any, List, Optional, Tuple
import fitz  # PyMuPDF  (used by write_pdf_description)

# --------- Human-readable meanings for final keys (edit/grow freely) ----------
FIELD_DOC: Dict[str, str] = {
    "report_year":        "Calendar year the report covers.",
    "filer_type":         "Type of filer (Individual/Entity/etc.).",
    "tin_type":           "Type of U.S. taxpayer ID (SSN/ITIN/EIN).",
    "tin_value":          "U.S. taxpayer ID number (format ###-##-#### for SSN/ITIN).",
    "foreign_id_type":    "Type of foreign identification (e.g., Passport).",
    "foreign_id_number":  "Foreign identification number.",
    "foreign_id_country": "Country that issued the foreign identification.",
    "dob":                "Date of birth (MM/DD/YYYY).",
    "first_name":         "First name of the filer (or contact).",
    "middle_initial":     "Middle initial (single character).",
    "last_or_org":        "Last name or organization name.",
    "address_line1":      "Mailing address line 1 (street).",
    "city":               "City (mailing address).",
    "state":              "State/Province (mailing address).",
    "postal":             "ZIP/Postal code.",
    "country":            "Country (mailing address).",
    "amended":            "Whether this is an amended filing (Yes/No).",
    "q14a_answer":        "Q14a: Financial interest in 25+ accounts (Yes/No).",
    "q14b_answer":        "Q14b: Signature authority over 25+ accounts (Yes/No).",
}

# ---------- label map → final keys (regex label, canonical key, value-hint) ----------
LABELS = [
    (r"\bTHIS REPORT IS FOR CALENDAR YEAR ENDED\b", "report_year", None),
    (r"\bTYPE OF FILER\b", "filer_type", None),
    (r"\bU\.S\. TAXPAYER IDENTIFICATION NUMBER\b", "tin_value", "ssn"),
    (r"\bTIN TYPE\b", "tin_type", None),
    (r"\bFOREIGN IDENTIFICATION\b", None, None),  # section header only
    (r"\b4A\b.*\bTYPE\b", "foreign_id_type", None),
    (r"\b4B\b.*\bNUMBER\b", "foreign_id_number", "passport"),
    (r"\b4C\b.*\bCOUNTRY OF ISSUE\b", "foreign_id_country", None),
    (r"\bINDIVIDUAL[’']S DATE OF BIRTH\b", "dob", "date"),
    (r"\bFIRST NAME\b", "first_name", None),
    (r"\bMIDDLE INITIAL\b", "middle_initial", None),
    (r"\bLAST NAME OR ORGANIZATION NAME\b", "last_or_org", None),
    (r"\bMAILING ADDRESS\b", "address_line1", None),
    (r"\bCITY\b", "city", None),
    (r"\bSTATE\b", "state", None),
    (r"ZIP/POSTAL CODE", "postal", "zip"),
    (r"\bCOUNTRY\b", "country", None),
    (r"\bAMENDED\b", "amended", None),
    (r"\b14A\b.*25 OR MORE", "q14a_answer", None),
    (r"\b14B\b.*25 OR MORE", "q14b_answer", None),
]

# ---------- value validators (hint → regex) ----------
RX = {
    "date": re.compile(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b"),
    "ssn": re.compile(r"\b\d{3}-?\d{2}-?\d{4}\b"),
    "zip": re.compile(r"\b\d{5}(?:-\d{4})?\b"),
    "passport": re.compile(r"\b[0-9A-Z]{6,15}\b"),
}

# Dot-leader cleanup, e.g. "Name . . . . . . " -> "Name "
DOTS = re.compile(r"\s(\.[\s\.]*){3,}")

def clean_text(s: str) -> str:
    """Normalize visual fillers and whitespace."""
    s = DOTS.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def lines_from_page(text_dict: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Flatten PyMuPDF text dict into lines with merged spans and keep bbox."""
    out = []
    for b in text_dict.get("blocks", []):
        for ln in b.get("lines", []):
            txt = clean_text("".join(sp.get("text", "") for sp in ln.get("spans", [])))
            if txt:
                out.append({"text": txt, "bbox": ln.get("bbox", [0,0,0,0])})
    return out

def y_center(bb): return (bb[1]+bb[3])/2.0
def same_band(a,b,y_tol=8): return abs(y_center(a)-y_center(b)) <= y_tol

def nearest_right(anchor_ln, lines, max_dx=220, y_tol=8):
    """
    Choose a value to the RIGHT of label on the same y-band (classic form layout).
    max_dx/y_tol are tuned heuristics; adjust for other templates.
    """
    ax1 = anchor_ln["bbox"][2]
    best, bestd = None, 1e9
    for ln in lines:
        x0 = ln["bbox"][0]
        if x0 >= ax1 and same_band(anchor_ln["bbox"], ln["bbox"], y_tol):
            d = x0 - ax1
            if 0 <= d <= max_dx and d < bestd:
                best, bestd = ln, d
    return best

def nearest_below(anchor_ln, lines, max_dy=40):
    """
    Fallback: choose a value BELOW the label, within a short vertical window.
    Useful when forms stack values underneath labels.
    """
    ay1 = anchor_ln["bbox"][3]
    best, bestd = None, 1e9
    for ln in lines:
        dy = ln["bbox"][1] - ay1
        if 0 <= dy <= max_dy:
            d = math.hypot((ln["bbox"][0]-anchor_ln["bbox"][0]), dy)
            if d < bestd:
                best, bestd = ln, d
    return best

def label_hits(lines):
    """Find all lines that look like field labels based on LABELS regex."""
    hits = []
    for pat, key, hint in LABELS:
        rx = re.compile(pat, re.I)
        for ln in lines:
            if rx.search(ln["text"]):
                hits.append((key, hint, ln))
    return hits

def confidence(anchor_ln, value_ln, hint):
    """
    Transparent confidence:
      base 0.5
      +0.2 if same y-band (rightward extraction)
      +up to 0.2 for proximity
      +0.1 if value passes regex validation for its hint
    """
    if not value_ln: 
        return 0.2
    base = 0.5
    band = 0.2 if same_band(anchor_ln["bbox"], value_ln["bbox"]) else 0.0
    d = max(1.0, value_ln["bbox"][0] - anchor_ln["bbox"][2])
    dist = min(0.2, 60.0 / (60.0 + d))
    rx_bonus = 0.0
    if hint and hint in RX and RX[hint].search(value_ln["text"]):
        rx_bonus = 0.1
    return round(base + band + dist + rx_bonus, 2)

def merge_address_lines(current_value, new_text):
    """
    Address line stitcher:
      - Appends short follow-up lines (APT/UNIT/etc.) to line1.
    """
    if not current_value:
        return new_text
    if len(new_text) <= 10 or re.search(r"\b(apt|unit|ste|#)\b", new_text, re.I):
        return current_value + " " + new_text
    return current_value + " " + new_text

def reduce_page(text_dict, include_provenance: bool=False, include_descriptions: bool=True) -> Dict[str, Any]:
    """
    Convert a single page's text_dict into {key: {value, confidence, [description, provenance...]}}.
    Keeps only the best (highest confidence) per key.
    """
    lines = lines_from_page(text_dict)
    out: Dict[str, Any] = {}

    hits = label_hits(lines)
    for key, hint, a_ln in hits:
        if key is None:
            continue  # label is a section header with no direct value

        # Find candidate value near the label
        cand = nearest_right(a_ln, lines) or nearest_below(a_ln, lines)
        conf = confidence(a_ln, cand, hint)
        val  = cand["text"] if cand else None

        # Special handling/tweaks
        if key == "report_year":
            # If value looks like "12/31/", scan page for a 4-digit year as fallback
            page_text = " ".join(ln["text"] for ln in lines)
            m = re.search(r"\b(19|20)\d{2}\b", page_text)
            if m: 
                val = m.group(0)
        if key == "amended":
            # Real value is a checkbox; default "No" unless you wire a checkbox detector
            val = "No"

        if key == "address_line1" and val:
            # Try to attach a short second line (APT/UNIT)
            below = nearest_below(cand, lines, max_dy=18)
            if below and below["text"] and len(below["text"]) <= 20:
                val = merge_address_lines(val, below["text"])

        rec = {"value": val, "confidence": conf}
        if include_descriptions and key in FIELD_DOC:
            rec["description"] = FIELD_DOC[key]
        if include_provenance:
            rec.update({"key_bbox": a_ln["bbox"], "value_bbox": (cand["bbox"] if cand else None)})

        # Keep the best-scoring value per key
        if key not in out or conf > out[key]["confidence"]:
            out[key] = rec

    # Minimal defaults for Yes/No questions (until checkbox detector is added)
    for qk in ("q14a_answer","q14b_answer"):
        if qk in out and not out[qk]["value"]:
            out[qk].update({"value": "No", "confidence": max(out[qk]["confidence"], 0.60)})
            if include_descriptions and qk in FIELD_DOC:
                out[qk]["description"] = FIELD_DOC[qk]

    return out

def reduce_document(capture: Dict[str, Any], include_provenance: bool=False, include_descriptions: bool=True) -> Dict[str, Any]:
    """
    Merge reduced pages: return highest-confidence record for each key across all pages.
    """
    merged: Dict[str, Any] = {}
    for page in capture["pages"]:
        page_out = reduce_page(page["text_dict"], include_provenance=include_provenance, include_descriptions=include_descriptions)
        for k, v in page_out.items():
            if k not in merged or v["confidence"] > merged[k]["confidence"]:
                merged[k] = v
    return merged

# --------------------- PDF summary writer ---------------------
def write_pdf_description(out_pdf_path: str, reduced: Dict[str, Any], title: str="Form Extraction Summary"):
    """
    Create a simple 1-page PDF summarizing the reduced JSON:
      Title
      Then rows: Key — Value — Confidence — Meaning
    Uses PyMuPDF, no extra deps.
    """
    doc = fitz.open()           # new empty PDF
    page = doc.new_page()       # 1st page
    width, height = page.rect.width, page.rect.height

    # Layout params
    x_key, x_val, x_conf, x_desc = 36, 180, 430, 480
    y = 48

    # Title
    page.insert_text((36, y), title, fontsize=14, fontname="helv", fill=(0,0,0))
    y += 24
    page.insert_text((36, y), "Key", fontsize=10, fontname="helv")
    page.insert_text((x_val, y), "Value", fontsize=10, fontname="helv")
    page.insert_text((x_conf, y), "Conf", fontsize=10, fontname="helv")
    page.insert_text((x_desc, y), "Description", fontsize=10, fontname="helv")
    y += 12
    page.draw_line((36, y), (width-36, y))
    y += 14

    # Rows (wrap long strings)
    def wrap(text: str, max_chars: int) -> List[str]:
        words = (text or "").split()
        lines, cur = [], []
        for w in words:
            if sum(len(x) for x in cur) + len(cur) - 1 + len(w) > max_chars:
                lines.append(" ".join(cur)); cur = [w]
            else:
                cur.append(w)
        if cur: lines.append(" ".join(cur))
        return lines or [""]

    for k in sorted(reduced.keys()):
        val = reduced[k].get("value")
        conf = f"{reduced[k].get('confidence', 0):.2f}"
        desc = reduced[k].get("description", FIELD_DOC.get(k, ""))

        key_lines  = wrap(k, 18)
        val_lines  = wrap(str(val), 34)
        desc_lines = wrap(desc, 40)
        max_lines = max(len(key_lines), len(val_lines), len(desc_lines), 1)

        for i in range(max_lines):
            page.insert_text((36,    y), key_lines[i]  if i < len(key_lines)  else "", fontsize=9, fontname="helv")
            page.insert_text((x_val, y), val_lines[i]  if i < len(val_lines)  else "", fontsize=9, fontname="helv")
            if i == 0:
                page.insert_text((x_conf, y), conf, fontsize=9, fontname="helv")
            page.insert_text((x_desc, y), desc_lines[i] if i < len(desc_lines) else "", fontsize=9, fontname="helv")
            y += 12
            if y > height - 48:  # new page if overflow
                page = doc.new_page()
                y = 48

    doc.save(out_pdf_path)
    doc.close()

# --------------------- Example run ---------------------
if __name__ == "__main__":
    # Load your capture JSON (output of capture_pdf)
    with open("/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/response.json","r") as f:
        capture = json.load(f)

    # Reduce → concise JSON (with descriptions)
    result = reduce_document(capture, include_provenance=False, include_descriptions=True)
    print(json.dumps(result, indent=2))

    # Also write a readable PDF summary next to it (optional)
    write_pdf_description(
        "/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/response_summary.pdf",
        result,
        title="FBAR (FinCEN 114) – Page Extraction Summary"
    )


{
  "report_year": {
    "value": "2024",
    "confidence": 0.9,
    "description": "Calendar year the report covers."
  },
  "filer_type": {
    "value": "Individual",
    "confidence": 0.9,
    "description": "Type of filer (Individual/Entity/etc.)."
  },
  "tin_value": {
    "value": "123654987",
    "confidence": 1.0,
    "description": "U.S. taxpayer ID number (format ###-##-#### for SSN/ITIN)."
  },
  "tin_type": {
    "value": "SSN/ITIN",
    "confidence": 0.9,
    "description": "Type of U.S. taxpayer ID (SSN/ITIN/EIN)."
  },
  "foreign_id_type": {
    "value": "Passport",
    "confidence": 0.9,
    "description": "Type of foreign identification (e.g., Passport)."
  },
  "foreign_id_country": {
    "value": "Falkland Islands (Isla",
    "confidence": 0.9,
    "description": "Country that issued the foreign identification."
  },
  "dob": {
    "value": "06/06/1979",
    "confidence": 1.0,
    "description": "Date of birth (MM/DD/YYYY)."
  },
  "first_name": {
    "value": "Fname